# Day 2 Homework — Web Summariser with Ollama

This is the Day 1 web summariser rebuilt to use a **local open-source model via Ollama** instead of OpenAI.

### What changes from Day 1
| Day 1 (OpenAI) | Day 2 (Ollama) |
|---|---|
| Requires `OPENAI_API_KEY` in `.env` | No API key needed |
| `OpenAI()` | `OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')` |
| Model: `gpt-4.1-mini` | Model: `llama3.2` |

Ollama exposes an OpenAI-compatible API locally, so the same Python client library works with just a different base URL.
Note: the variable is named `ollama` (not `openai`) to make clear we are talking to a local Ollama server, not OpenAI's cloud.

### Prerequisites
1. Install Ollama from https://ollama.com
2. Pull the model: `ollama pull llama3.2`
3. Ollama runs automatically in the background once installed — no need to start it manually

In [1]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

## Connect to Ollama

No `.env` file or API key needed — Ollama runs locally on port 11434.
We use the OpenAI client library as a wrapper since Ollama exposes an OpenAI-compatible endpoint.

In [2]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

MODEL = 'llama3.2'

## Quick sanity check

In [3]:
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Hello! Reply in one sentence.'}]
)
print(response.choices[0].message.content)

I'm happy to help you with any questions or information you need, and I'll do my best to provide a helpful response.


## Website scraper

Same approach as Day 1 — `requests` + `BeautifulSoup` to strip out scripts, styles and navigation noise.

In [4]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}

def fetch_website_contents(url):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    title = soup.title.string if soup.title else 'No title found'
    if soup.body:
        for tag in soup.body(['script', 'style', 'img', 'input']):
            tag.decompose()
        text = soup.body.get_text(separator='\n', strip=True)
    else:
        text = ''
    return (title + '\n\n' + text)[:5_000]

## Prompts

In [5]:
system_prompt = """
You are an assistant that analyzes the contents of a website and provides
a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block.
"""

def messages_for(url):
    contents = fetch_website_contents(url)
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'Here are the contents of a website. Provide a short summary.\n\n{contents}'}
    ]

## Summarise function

In [6]:
def summarize(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(url)
    )
    return response.choices[0].message.content

def display_summary(url):
    display(Markdown(summarize(url)))

## Try it out

In [7]:
display_summary('https://edwarddonner.com')

**Summary**

The website is owned by Edward Donner, the co-founder and CTO of Nebula.io. He's a passionate AI enthusiast who shares his knowledge through courses on Udemy, which have gained massive success (specifically 500,000 enrolled across 194 countries). The website also features an "Outsmart" arena where language models battle each other in a diplomacy-themed game, as well as other AI-related content, such as posts and resources. Edward's personal story and his involvement with various AI projects are also covered on the site.

In [8]:
display_summary('https://bbc.com/news')

**Summary**

Recent news articles discuss various global topics. 

* A deal between the US and Iran to potentially end tensions is in progress, but details are not yet clear.
* Oil prices have decreased due to hopes of a peace agreement with Iran.
* Hong Kong astronaut Lee Yea-ek has launched into space as part of a Chinese mission, setting records for individual swimmers.
* A large-scale Russian attack on Ukraine resulted in the death of four people and many injuries.
* China's deadliest coal mining disaster in years occurred, leading to outrage among authorities and security concerns for the environment. 
* Clashes broke out at Venezuelan prisons due to alleged mistreatment by authorities.
* Arsenal has finally won the Premier League title, marking a 22-year wait, while Indian billionaires have spent major amounts on foreign companies due to slowing domestic growth.

Note that news articles are subject to change and may be updated or superseded over time.

In [9]:
display_summary('https://anthropic.com')

# Anthropic Website Summary
Anthropic is a public benefit corporation working on securing the benefits and mitigating the risks of AI. The company focuses on building responsible AI solutions, prioritizing safety at the forefront.

## Products and Services
- Claude: an AI platform designed for development and complex professional work.
- Myths: coming soon AI agent project.
- Claudia Code: enterprise code modernization solution.
- Claude Platform: customizable software development environment.

## Research and Policy
Anthropic conducts research on Economic Futures, Alignment Science, and Transparency. They also publish the Responsible Scaling Policy and Claude's Constitution to guide their approach to secure AI benefits.

## Community Engagement
The company engages in conversations and partnerships through forums like Claude 'Where everyone learns.'

## Try a different model

Ollama supports many models. To try another one:
```bash
ollama pull mistral
ollama pull deepseek-r1
```
Then just change the `MODEL` variable above and re-run.